# Case study: a project that runs late {#sec-case-project}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c15-project.ipynb)

A software team of 20 developers is asked to build a new booking system for a hospital group. The scope is fixed at 1,000 tasks, and the plan is simple arithmetic: 20 people, each finishing one task a week, need 50 weeks and 1,000 person-weeks. The project reports steady progress for most of a year. Then, at about 90% complete, progress stalls. The deadline passes, people are added, overtime becomes normal, and the system is finally delivered months late and far over budget. The same story is told about ships, power stations, aircraft and office buildings; only the units change.

This chapter is a case study. It runs the whole modelling process of the book on one problem, in compressed form: the problem and its reference mode, a dynamic hypothesis and a boundary, a model built from molecules, tests, an explanation of the behaviour, policy experiments with a check of robustness, and a brief for the person who has to decide. The sketch after the setup cell shows the structure we will build.

**Who it is for.** Analysts who are asked why a project is late and what would help, and anyone who wants to see the steps of the earlier chapters applied end to end. **Who can skip.** Readers who already know project models with a rework cycle can skim sections 1 to 3 and start at section 4. **Prerequisites.** Chapter 2, From story to diagram (reference modes, boundaries and dynamic hypotheses); chapter 5, Decisions inside the model (decision rules and the Doing Work molecule); chapter 6, Why things swing (the Workforce molecule and hiring delays); chapter 7, Patterns that repeat (overtime, fatigue, quality, rework and schedule pressure); chapter 9, Testing your model; and chapter 11, From model to decision (policies, robustness and the brief). **Time.** About three hours with the exercises.

All project data in this chapter are synthetic, generated in the code with a fixed seed.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package,
# which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 15
rng = np.random.default_rng(SEED)

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__)

In [ ]:
#| fig-alt: "Stock-and-flow sketch of the project. Work to do flows to Work done as correct work, or up to Undiscovered rework, which flows back to Work to do when the flaws are found. Below, posts are opened, filled by hiring into New staff, who become Experienced staff. On the right, Fatigue builds from overtime. Dashed arrows show schedule pressure driving overtime and hiring, staff setting the pace of work, and fatigue, haste and new staff lowering quality."
from matplotlib.patches import Circle, FancyBboxPatch


def draw_sfd(ax, stocks, flows, links):
    """A small stock-and-flow diagram (as in chapter 7)."""
    w, h = 1.7, 0.8
    for name, (x, y) in stocks.items():
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                    boxstyle="round,pad=0.02", fc="white",
                                    ec="black", lw=1.5))
        ax.text(x, y, name, ha="center", va="center", fontsize=9)

    def inside(p):
        return any(abs(p[0] - x) <= w / 2 + 1e-9
                   and abs(p[1] - y) <= h / 2 + 1e-9
                   for x, y in stocks.values())

    for label, a, b, *where in flows:
        for p in (a, b):
            if not inside(p):
                ax.text(*p, "☁", ha="center", va="center",
                        fontsize=18, color="grey")
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="-|>", lw=3, color="black",
                                    shrinkA=8, shrinkB=2))
        mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
        ax.add_patch(Circle((mx, my), 0.12, fc="white", ec="black",
                            lw=1.2, zorder=3))
        dx, dy = where[0] if where else (0, -0.3)
        ax.text(mx + dx, my + dy, label, ha="center" if not dx else
                ("left" if dx > 0 else "right"), va="top" if dy < 0 else
                "center", fontsize=8)
    for a, b, label, rad, where in links:
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", lw=1, ls="--",
                                    color="#0072B2", shrinkA=4, shrinkB=4,
                                    connectionstyle=f"arc3,rad={rad}"))
        ax.text(*where, label, ha="center", fontsize=8, color="#0072B2")
    ax.set_aspect("equal")
    ax.axis("off")


fig, ax = plt.subplots(figsize=(9.5, 5.2))
draw_sfd(ax,
         stocks={"Work to do": (3, 3), "Undiscovered\nrework": (3, 5.6),
                 "Work done": (8.5, 3), "Posts open": (3, 0),
                 "New staff": (6, 0), "Experienced\nstaff": (9, 0),
                 "Fatigue": (11.5, 5.6)},
         flows=[("Correct work", (3.85, 3), (7.65, 3), (0, 0.3)),
                ("Creating rework", (3.4, 3.4), (3.4, 5.2), (0.25, 0)),
                ("Discovering", (2.6, 5.2), (2.6, 3.4), (-0.25, 0)),
                ("Opening posts", (0.9, 0), (2.15, 0)),
                ("Hiring", (3.85, 0), (5.15, 0)),
                ("Assimilating", (6.85, 0), (8.15, 0)),
                ("Getting fatigued", (13.4, 5.6), (12.35, 5.6))],
         links=[((3.9, 2.7), (1.6, 0.15), "behind: hire", 0.35, (1.2, 1.5)),
                ((7.5, 0.4), (5.9, 2.85), "capacity", -0.2, (7.3, 1.6)),
                ((3.9, 3.3), (11.0, 5.2), "pressure:\novertime, haste",
                 -0.25, (8.6, 4.45)),
                ((10.7, 5.6), (3.5, 4.35), "tired, rushed and new\n"
                 "staff make more errors", 0.08, (7.2, 5.85))])
ax.set_xlim(0.4, 14.3)
ax.set_ylim(-0.9, 6.4)
ax.set_title("A fixed-scope project with rework, hiring and fatigue")
plt.show()

Figure: The project as stocks and flows. Work moves from `Work to do` to `Work done`, but a share of it is done with flaws and waits, unseen, in `Undiscovered rework` until testing finds it; below, staff move from open posts through new hires to experienced staff, and on the right fatigue builds from overtime. Notice that the dashed links all start from the same place: when the work to do looks too large for the time left, managers push for overtime, accept haste and hire, and each of these feeds errors back into the work.

**Learning outcomes.** By the end of the chapter you can:

1. **read** the reference mode of a late project from status reports and **describe** the "90% done" syndrome in numbers (Try it 1);
2. **state** a dynamic hypothesis for a project overrun and **classify** its feedback loops (Try it 2);
3. **assemble** a project model from molecules, one sector at a time (Try it 3);
4. **verify** the model with conservation, extreme-condition and time-step tests, and **repair** a fault the tests find (Try it 4);
5. **diagnose** which loops and delays cause the overrun by switching mechanisms off one at a time (Try it 5);
6. **evaluate** policies across uncertain parameters and **justify** a recommendation in a short brief (Try it 6 and Tell the decision maker).

## 1. The problem and the reference mode

A model starts from a pattern over time that someone wants explained, the **reference mode** of chapter 2. For a project, the evidence is usually a set of weekly status reports: the percentage of the scope the team reports as complete, and the number of people on the project. We have no real project to hand, so the cell below makes up the reports of one. They are synthetic: a steady pace of 1.85 percentage points a week until week 46, then a crawl towards 100%, with a little noise; and staff that stay near 20 until week 30 and then rise to about 38. The shape is the one that project managers describe again and again, and that is all we ask of it here.

In [ ]:
# SYNTHETIC status reports for one project, made up for this chapter.
# Reported progress climbs at a steady pace, then crawls; staff are added
# late. Seeded, so every reader gets the same numbers.
weeks = np.arange(0, 77)
pace = 1.85                                  # percentage points a week
trend = np.where(weeks <= 46, pace * weeks,  # np.where: if, else by element
                 100 - (100 - pace * 46) * np.exp(-(weeks - 46) / 8))
noisy = trend + rng.normal(0, 0.8, len(weeks))
# a running maximum: a status report rarely admits that progress went down
reported = np.clip(np.maximum.accumulate(noisy), 0, 100)
staff = 20 + 18 / (1 + np.exp(-(weeks - 44) / 4))
staff = np.round(staff + rng.normal(0, 0.4, len(weeks))).clip(min=20)
staff = staff.astype(int)
reports = pd.DataFrame({"reported progress": reported.round(1),
                        "staff": staff},
                       index=pd.Index(weeks, name="week"))
PLAN_WEEKS, PLAN_EFFORT = 50, 1000           # the naive plan: 20 people x 50
plan = np.minimum(100, 100 * weeks / PLAN_WEEKS)

rep = reports["reported progress"]
w90 = rep.index[rep >= 90][0]
w_end = rep.index[rep >= 99.5][0]
print(reports.iloc[::8].T.to_string())
print(f"\nreported 90% in week {w90}; delivered (99.5%) in week {w_end}; "
      f"the last 10% took {w_end - w90} weeks, "
      f"{(w_end - w90) / w_end:.0%} of the project")
print(f"staff: {staff[0]} at the start, {staff.max()} at the peak; "
      f"effort {staff.sum():,} person-weeks against {PLAN_EFFORT:,} planned")

In [ ]:
#| fig-alt: "Two panels over 76 weeks of synthetic status reports. Left, reported progress against the plan: it tracks slightly below the straight-line plan to about 85% in week 46, then crawls towards 100% until about week 66, long after the planned week 50. Right, staff on the project: 20 until week 30, then rising steeply to about 38 by week 55."
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(weeks, plan, label="plan")
a1.plot(weeks, rep, label="reported (synthetic)")
a1.axvline(PLAN_WEEKS, color="grey", lw=1, ls=":")
a1.set(title="Reported progress", xlabel="time (week)",
       ylabel="progress (% of scope)")
a2.step(weeks, staff, where="post", color="#D55E00", ls="--")
a2.set(title="Staff on the project", xlabel="time (week)",
       ylabel="staff (people)", ylim=(0, 42))
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: Synthetic weekly status reports for one project, against the naive plan of 2% of the scope a week (left), and the staff on the project (right). Reported progress keeps close to the plan for 40 weeks and then flattens; the staff rise steeply only once the project is visibly late.

The reports tell a familiar story. Reported progress reaches 90% in week 50, the planned end, and the system is delivered (reported at 99.5% or more) in week 66. The last 10% took 16 weeks, 24% of the project's life: this is the **"90% done" syndrome**, in which a project reports itself nearly finished for months. Staff rose from 20 to a peak of 39, and the project used 2,137 person-weeks against the 1,000 planned. The reference mode, then, has three parts: progress that tracks the plan and then stalls, staff added late, and a cost overrun much larger than the schedule overrun. A good model of this project should produce all three, and for the right reasons.

**Try it 1.**

- *Check.* What share of the project's life went on the last 10% of the reported progress?
- *Apply.* In week 40 a manager fits a straight line to the reports of weeks 10 to 40 and extrapolates to 100%. Which week does the line predict, and how wrong is it?
- *Extend.* By how much, in percent, did the effort exceed the plan? Why is the cost overrun so much larger than the schedule overrun?

<details><summary>Worked answer</summary>

The last 10% took 16 of 66 weeks, 24%. The pace of weeks 10 to 40 is 1.83 points a week, and the straight line reaches 100% in week 54.4, almost 12 weeks before the real delivery in week 66. Trend extrapolation from reported progress is the method most status reports use, and it fails exactly where it matters, because the reports do not count the work that will have to be done again. The effort was 2,137 person-weeks, 114% over plan, against a schedule overrun of 32% (66 weeks against 50). Cost grows faster than time because the late weeks are worked by a bigger team, often with overtime.
</details>

In [ ]:
# Check: the share of the project spent on the last 10%
print(f"last 10%: {w_end - w90} of {w_end} weeks = "
      f"{(w_end - w90) / w_end:.0%}")
# Apply: in week 40, extrapolate the pace of weeks 10 to 40
early = rep.loc[10:40]
slope, intercept = np.polyfit(early.index, early.values, 1)
print(f"pace weeks 10 to 40: {slope:.2f} points a week; forecast finish "
      f"week {(100 - intercept) / slope:.1f}; actual week {w_end}")
# Extend: compare effort with the plan
print(f"effort: {staff.sum():,} person-weeks, "
      f"{staff.sum() / PLAN_EFFORT - 1:.0%} over plan")

## 2. Dynamic hypothesis and boundary

A **dynamic hypothesis** (chapter 2) names the feedback structure that we think produces the reference mode. Ours, in four sentences:

> Part of the work is done with flaws, and the flaws stay hidden until testing finds them, so reported progress runs ahead of true progress and the work to do is underestimated until late. As the deadline nears, the visible work looks too large for the time left; managers respond with overtime, haste and hiring, which raise output at first. Overtime builds fatigue, haste and fatigue lower quality, and new hires work slowly, take time from experienced staff and make more errors, so each response also creates more hidden rework or less capacity. Because the rework surfaces only after a delay, these side effects arrive after the deadline, when the project looked nearly finished.

The causal links behind this story fit in a short edge list. The function from chapter 2 finds every loop in it and its polarity: an even number of negative links makes a reinforcing loop (R), an odd number a balancing one (B).

In [ ]:
# The causal links of the dynamic hypothesis (cause, effect, polarity)
edges = pd.DataFrame([
    ("Work to do", "Schedule pressure", "+"),
    ("Schedule pressure", "Overtime", "+"),
    ("Overtime", "Work rate", "+"),
    ("Work rate", "Work to do", "-"),
    ("Overtime", "Fatigue", "+"),
    ("Fatigue", "Capacity", "-"),
    ("Capacity", "Work rate", "+"),
    ("Capacity", "Schedule pressure", "-"),
    ("Schedule pressure", "Haste", "+"),
    ("Haste", "Work rate", "+"),
    ("Haste", "Quality", "-"),
    ("Fatigue", "Quality", "-"),
    ("Quality", "Undiscovered rework", "-"),
    ("Undiscovered rework", "Work to do", "+"),
    ("Schedule pressure", "New staff", "+"),
    ("New staff", "Experienced staff", "+"),
    ("Experienced staff", "Capacity", "+"),
    ("New staff", "Overheads", "+"),
    ("Overheads", "Capacity", "-"),
    ("New staff", "Quality", "-"),
], columns=["cause", "effect", "sign"])


def find_loops(edges):
    """Every simple loop and its polarity (the function of chapter 2)."""
    out = {}
    for cause, effect, sign in edges.itertuples(index=False):
        out.setdefault(cause, []).append((effect, sign))
    loops = []

    def follow(start, here, path, signs):
        for nxt, sign in out.get(here, []):
            if nxt == start:
                negatives = (signs + [sign]).count("-")
                loops.append((path, "R" if negatives % 2 == 0 else "B"))
            elif nxt > start and nxt not in path:
                follow(start, nxt, path + [nxt], signs + [sign])

    for start in sorted(set(edges["cause"]) | set(edges["effect"])):
        follow(start, start, [start], [])
    return loops


loops = find_loops(edges)
print(len(loops), "loops")
for path, polarity in sorted(loops, key=lambda x: (x[1], len(x[0]))):
    print(polarity, " -> ".join(path + [path[0]]))

Eleven loops, four balancing and seven reinforcing. Several come in pairs that tell the same story by two routes (through `Capacity` and pressure directly, or through the work rate and the work to do), so they group into eight named loops:

| Loop | Polarity | Story | Delay on the way |
|---|---|---|---|
| Work harder | B | pressure raises overtime, which raises the work rate and cuts the work to do | none |
| Rush | B | pressure raises haste, which raises the work rate | none |
| Hire | B (two routes) | pressure raises hiring; new staff become experienced and add capacity | hiring and assimilation |
| Burnout | R (two routes) | overtime builds fatigue, which cuts capacity and raises pressure | fatigue |
| Haste makes waste | R | haste lowers quality; flaws come back as work to do | rework discovery |
| Tired errors | R | fatigue lowers quality; flaws come back and raise pressure and overtime | fatigue, rework discovery |
| Too many cooks | R (two routes) | new staff bring training and communication overheads that cut capacity | none |
| Green errors | R | new staff make more errors, which come back as work to do | rework discovery |

The balancing loops are the ones managers intend; the reinforcing ones are their side effects. Notice where the delays are. The intended loops act within weeks, except hiring. Every loop that runs through quality has to pass through `Undiscovered rework`, which releases its contents only after the time it takes to find a flaw. That is the hypothesis in one line: the cures act now and their costs arrive later, after the work has been reported as done.

The **model boundary** (chapter 2) says what the model works out, what it takes as given, and what it leaves out:

| Endogenous (worked out) | Exogenous (given) | Excluded, and why |
|---|---|---|
| work to do, undiscovered rework, work done | project scope, 1,000 tasks | scope changes: the case is a fixed-scope contract (the Extend below adds them) |
| schedule pressure, overtime, haste | deadline, week 50 | staff who quit from fatigue: the project is short next to typical tenure |
| fatigue, quality, rework discovery | normal productivity and quality | slack when ahead of schedule: this project is never ahead |
| open posts, new and experienced staff | hiring delays, maximum staff of 40 | re-planning the deadline: we test deadlines as policies instead |
| effort spent | share of time on quality assurance | morale, budgets and penalties: second-order for the schedule |

**Try it 2.**

- *Check.* Which loops in the table are balancing, and which of them carries a long delay?
- *Apply.* The `Hire` loop is balancing, but `Too many cooks` and `Green errors` start from the same hiring decision. Predict, before section 5, whether hiring late will shorten or lengthen the project.
- *Extend.* Customers often add features when a project looks late, hoping to get them in before the release. Add the links `Schedule pressure -> Scope added (+)` and `Scope added -> Work to do (+)`. What new loop appears, and what is its polarity?

<details><summary>Worked answer</summary>

*Check.* Work harder, Rush and Hire (by two routes) are balancing; only Hire carries a long delay, through hiring and assimilation. *Apply.* Early hiring should help, because the new staff have time to become experienced; late hiring adds overheads and errors when there is little time left for the newcomers to pay them back, so it may lengthen the project. Section 6 tests this, and the answer depends on the timing. *Extend.* One new loop: Schedule pressure -> Scope added -> Work to do -> Schedule pressure, with no negative links, so it is reinforcing. Pressure attracts more scope, which raises the pressure: a project that looks late invites the changes that make it later.
</details>

In [ ]:
# Extend: scope creep. Customers add work when the project looks late
creep = pd.concat([edges, pd.DataFrame(
    [("Schedule pressure", "Scope added", "+"),
     ("Scope added", "Work to do", "+")], columns=edges.columns)])
new = [l for l in find_loops(creep) if "Scope added" in l[0]]
for path, polarity in new:
    print(polarity, " -> ".join(path + [path[0]]))

## 3. The model in code

We build the model in three steps, each a list of stocks, flows and variables, with the `build` helper of chapter 5: a later part replaces any element of an earlier part with the same name, so a constant in step 1 can become a rule in step 2. The parts are our own implementations of structures that Hines describes in *Molecules of Structure* [@hines_molecules]; the cell below lists the molecules from chapters 5, 6 and 7 that we draw on, and prints the one at the centre of the case. Appendix B (@sec-package) documents the spec format and the equation language.

In [ ]:
# The molecules the model is built from, as shipped with the package
for key in ("reworkCycle", "doingWork", "schedulePressure", "overtime",
            "effectFatigue", "prodFatigue", "quality", "workforce"):
    m = lb.molecule(key)
    print(f"{key:<17} {m.name:<24} {len(m.stocks)} stocks, "
          f"{len(m.flows)} flows, {len(m.vars)} variables")
print()
print(lb.molecule("reworkCycle").equations())

**Step 1: the work.** The **Rework Cycle** (`reworkCycle`), after Hines, *Molecules of Structure* [@hines_molecules], is the heart of the model. Work is accomplished at a rate $A$; a fraction $q$, the **quality**, is done correctly and goes to `Work done`; the rest goes to `Undiscovered rework` $U$, which returns to `Work to do` after an average **rework discovery time** $T_D$:

$$
\text{Correct work} = A\,q, \qquad \text{Creating rework} = A\,(1 - q), \qquad \text{Discovering rework} = \frac{U}{T_D}.
$$

The rate $A$ is protected as in **Doing Work** (`doingWork`), after Hines, *Molecules of Structure* [@hines_molecules]: never more than the work to do divided by a minimum task time of half a week. Two quantities matter for reporting. **Reported progress** counts everything that has been worked on, $(\text{Work done} + U)$ over the scope, because nobody knows yet which tasks are flawed; **true progress** counts only `Work done`. We also make quality assurance (QA) explicit: a share of staff time, 5% by default, goes on reviews and tests, and more QA finds flaws sooner, $T_D = 12 / (1 + \text{QA share} / 0.05)$ weeks, so 6 weeks at the default. In step 1 the team is a fixed 20 people at normal quality 0.85, with no deadline. `finish_week` reads off the week the work truly done reaches 99% of the scope, which is our definition of delivery.

In [ ]:
def S(name, init, units):
    return dict(kind="stock", name=name, init=init, units=units)


def F(name, eq, units, src=None, dst=None):
    return dict(kind="flow", name=name, eq=eq, units=units, src=src, dst=dst)


def V(name, eq, units):
    return dict(kind="var", name=name, eq=eq, units=units)


def build(*parts, name="Software project", T=150, dt=0.125):
    """A model from lists of S, F and V (as in chapter 5); later parts
    replace earlier elements that have the same name."""
    els = {}
    for part in parts:
        for e in part:
            els[e["name"]] = e
    key = {n: n.lower().replace(" ", "_") for n in els}
    stocks, flows, vars_ = [], [], []
    for n, e in els.items():
        if e["kind"] == "stock":
            stocks.append({"id": key[n], "name": n, "init": e["init"],
                           "units": e["units"], "nonNegative": True})
        elif e["kind"] == "flow":
            f = {"id": key[n], "name": n, "eq": e["eq"], "units": e["units"]}
            if e["src"]:
                f["from"] = key[e["src"]]
            if e["dst"]:
                f["to"] = key[e["dst"]]
            flows.append(f)
        else:
            vars_.append({"id": key[n], "name": n, "eq": e["eq"],
                          "units": e["units"]})
    return lb.Model.from_spec({"name": name, "timeUnit": "week", "T": T,
                               "dt": dt, "stocks": stocks, "flows": flows,
                               "vars": vars_})


def finish_week(d, scope=1000, share=0.99):
    """First week in which the work truly done reaches 99% of the scope."""
    hit = d.index[d["Work done"] >= share * scope]
    return hit[0] if len(hit) else np.nan


# Step 1: the Rework Cycle with the Doing Work protection, a team of 20
WORK = [
    S("Work to do", "Project_scope", "tasks"),
    S("Undiscovered rework", "0", "tasks"),
    S("Work done", "0", "tasks"),
    F("Correct work", "Accomplishing * Quality", "tasks/week",
      src="Work to do", dst="Work done"),
    F("Creating rework", "Accomplishing * (1 - Quality)", "tasks/week",
      src="Work to do", dst="Undiscovered rework"),
    F("Discovering rework", "Undiscovered_rework / Rework_discovery_time",
      "tasks/week", src="Undiscovered rework", dst="Work to do"),
    V("Project scope", "1000", "tasks"),
    V("Normal productivity", "1", "tasks/person/week"),
    V("QA share", "0.05", "fraction"),
    V("Normal capacity", "20 * Normal_productivity * (1 - QA_share)",
      "tasks/week"),
    V("Capacity in use", "Normal_capacity", "tasks/week"),
    V("Minimum task time", "0.5", "week"),
    V("Accomplishing",
      "MIN(Capacity_in_use, Work_to_do / Minimum_task_time)", "tasks/week"),
    V("Normal quality", "0.85", "fraction"),
    V("Quality", "Normal_quality", "fraction"),
    V("Discovery time without QA", "12", "week"),
    V("QA reference share", "0.05", "fraction"),
    V("Rework discovery time", "Discovery_time_without_QA"
      " / (1 + QA_share / QA_reference_share)", "week"),
    V("Reported progress",
      "100 * (Work_done + Undiscovered_rework) / Project_scope", "percent"),
    V("True progress", "100 * Work_done / Project_scope", "percent"),
]
step1 = build(WORK, name="Work only")
d1 = step1.run()
r1 = d1["Reported progress"]
print(f"quality 0.85, discovery time "
      f"{d1['Rework discovery time'].iloc[0]:.0f} weeks: reported 90% in "
      f"week {r1.index[r1 >= 90][0]:.1f}, true 99% in week "
      f"{finish_week(d1):.1f}")
d1q = step1.run(normal_quality=1)
print(f"quality 1: true 99% in week {finish_week(d1q):.1f}")
print(f"undiscovered rework peaks at {d1['Undiscovered rework'].max():.0f} "
      f"tasks in week {d1['Undiscovered rework'].idxmax():.1f}")

Even with no pressure at all, the work-only model shows the syndrome. Reported progress reaches 90% in week 54.8, but true progress reaches 99% only in week 65.2; with perfect quality the same team would finish in week 52.1. The hidden stock is small (it peaks at 17 tasks in week 60.4), but it is the last thing to empty, and it empties slowly, at a rate proportional to what is left.

**Step 2: pressure.** Now give the team a deadline in week 50. **Scheduled Completion and Pressure** (`schedulePressure`), after Hines, *Molecules of Structure* [@hines_molecules], gives the **schedule pressure** $P$: the rate needed to finish the visible work to do $W$ in the time left, over the normal capacity $C_n$,

$$
P = \frac{W / \max(\text{Deadline} - t,\ 2)}{C_n}.
$$

Two responses follow. **Overtime** (`overtime`), after Hines, *Molecules of Structure* [@hines_molecules], is the pressure, but never below 1 and never above a limit of 1.3. **Haste** is the part of the pressure that overtime does not absorb, $\min(1.3, \max(1, P / \text{Overtime}))$: people cut corners and work faster. Fatigue is a smooth of overtime over 4 weeks, as in **Effect of Fatigue** (`effectFatigue`), after Hines, *Molecules of Structure* [@hines_molecules], and the capacity actually used is the normal capacity times overtime, haste and the effect of fatigue on productivity, as in **Productivity, Overtime and Fatigue** (`prodFatigue`). Quality follows **Quality** (`quality`), after Hines, *Molecules of Structure* [@hines_molecules]: normal quality times effects of fatigue, haste and skill, capped at 1. Our versions use straight-line effects with caps, as in chapter 7, where Hines draws lookup curves; and our quality responds to haste rather than to schedule pressure itself, so that overtime alone harms quality only through fatigue.

In [ ]:
# Step 2: a deadline, schedule pressure, overtime, haste, fatigue, quality
PRESSURE = [
    S("Fatigue", "1", "dimensionless"),
    F("Getting fatigued", "(Overtime - Fatigue) / Time_to_fatigue",
      "1/week", dst="Fatigue"),
    V("Deadline", "50", "week"),
    V("Minimum time left", "2", "week"),
    V("Time left", "MAX(Deadline - TIME, Minimum_time_left)", "week"),
    V("Required rate", "Work_to_do / Time_left", "tasks/week"),
    V("Schedule pressure", "Required_rate / MAX(Normal_capacity, 0.001)",
      "dimensionless"),
    V("Overtime limit", "1.3", "dimensionless"),
    V("Overtime", "MAX(1, MIN(Overtime_limit, Schedule_pressure))",
      "dimensionless"),
    V("Haste limit", "1.3", "dimensionless"),
    V("Haste", "MAX(1, MIN(Haste_limit, Schedule_pressure / Overtime))",
      "dimensionless"),
    V("Time to fatigue", "4", "week"),
    V("Fatigue strength", "0.6", "dimensionless"),
    V("Effect of fatigue on productivity",
      "MAX(0.3, MIN(1.2, 1 - Fatigue_strength * (Fatigue - 1)))",
      "dimensionless"),
    V("Capacity in use", "Normal_capacity * Overtime * Haste"
      " * Effect_of_fatigue_on_productivity", "tasks/week"),
    V("Effect of fatigue on quality",
      "MAX(0.4, MIN(1.1, 1 - 0.4 * (Fatigue - 1)))", "dimensionless"),
    V("Effect of haste on quality",
      "MAX(0.4, MIN(1.1, 1 - 0.3 * (Haste - 1)))", "dimensionless"),
    V("Effect of skill on quality", "1", "dimensionless"),
    V("Quality", "MAX(0, MIN(1, Normal_quality"
      " * Effect_of_fatigue_on_quality * Effect_of_haste_on_quality"
      " * Effect_of_skill_on_quality))", "fraction"),
]
step2 = build(WORK, PRESSURE, name="Work under pressure, team of 20")
d2 = step2.run()
print(f"team of 20 with a deadline: true 99% in week {finish_week(d2):.1f}"
      f" (work only: {finish_week(d1):.1f})")
print(f"schedule pressure peaks at {d2['Schedule pressure'].max():.2f}; "
      f"overtime at its limit from week "
      f"{d2.index[d2['Overtime'] >= 1.3 - 1e-9][0]:.1f}; fatigue "
      f"{d2['Fatigue'].max():.2f}; quality falls to {d2['Quality'].min():.3f}")

With a fixed team of 20 and a deadline, the project now finishes in week 67.8, later than with no deadline at all (65.2). The plan asked for 20 tasks a week from a team that can deliver 19 at most, so pressure is above 1 from the first day; it peaks at 3.98, overtime sits at its limit from week 31.5, fatigue rises to 1.30, and quality falls from 0.85 to 0.681. The deadline makes the team work harder and finish later.

**Step 3: staff.** Managers who see the project slipping hire. The staff that the deadline seems to need are the work to do divided by the time left and normal productivity; the **desired staff** are that number, never below the 20 already on the project and never above a maximum of 40. Hiring follows the **Workforce** molecule (`workforce`) of chapter 6, after Hines, *Molecules of Structure* [@hines_molecules], with the open posts counted, as in the stock management rule of chapter 6: posts are opened over 2 weeks, filled over 6, and the new staff become experienced after an **assimilation time** of 16 weeks, a two-stage ageing chain as in chapter 4. New staff cost the project in three ways:

- they work at half the pace of experienced staff (`Rookie productivity`, 0.5);
- each takes a quarter of an experienced person's time to train (`Training overhead`, 0.25);
- they make more errors: average skill enters quality through the skill effect of the Quality molecule, with a new hire's skill at 0.5.

And everyone pays a **communication overhead** of 0.5% of their time for every other person on the team, so a team of 40 spends almost a fifth of its time coordinating. Two switches serve the policy tests later: `Responsive hiring` (1 means managers hire to the deadline, 0 means they do not) and a one-off request for `Staff added` people in a chosen week. Finally, `Effort spent` adds up the person-weeks worked, including overtime, until the work is delivered.

In [ ]:
# Step 3: staff who are hired late, learn slowly and talk to each other
STAFF = [
    S("Posts open", "0", "people"),
    S("New staff", "0", "people"),
    S("Experienced staff", "Initial_staff", "people"),
    S("Effort spent", "0", "person-weeks"),
    F("Opening posts", "MAX(0, Desired_staff - Total_staff - Posts_open)"
      " / Time_to_open_posts", "people/week", dst="Posts open"),
    F("Hiring", "Posts_open / Time_to_fill_posts", "people/week",
      src="Posts open", dst="New staff"),
    F("Assimilating", "New_staff / Assimilation_time", "people/week",
      src="New staff", dst="Experienced staff"),
    F("Spending effort", "Total_staff * Overtime * Project_active",
      "person-weeks/week", dst="Effort spent"),
    V("Initial staff", "20", "people"),
    V("Total staff", "New_staff + Experienced_staff", "people"),
    V("Rookie productivity", "0.5", "fraction"),
    V("Training overhead", "0.25", "people/person"),
    V("Training drain", "MIN(Experienced_staff,"
      " Training_overhead * New_staff)", "people"),
    V("Communication overhead per person", "0.005", "fraction/person"),
    V("Communication overhead", "MIN(0.5, Communication_overhead_per_person"
      " * MAX(0, Total_staff - 1))", "fraction"),
    V("Effective staff", "(Experienced_staff - Training_drain"
      " + Rookie_productivity * New_staff) * (1 - Communication_overhead)",
      "people"),
    V("Normal capacity", "Effective_staff * Normal_productivity"
      " * (1 - QA_share)", "tasks/week"),
    V("Rookie skill", "0.5", "fraction"),
    V("Average skill", "(Experienced_staff + Rookie_skill * New_staff)"
      " / MAX(Total_staff, 0.001)", "fraction"),
    V("Effect of skill on quality",
      "MAX(0.4, MIN(1.1, 0.4 + 0.6 * Average_skill))", "dimensionless"),
    V("Staff needed", "Work_to_do / (Time_left * Normal_productivity)",
      "people"),
    V("Responsive hiring", "1", "dimensionless"),
    V("Maximum staff", "40", "people"),
    V("Staff added", "0", "people"),
    V("Week staff added", "0", "week"),
    V("Desired staff", "MIN(Maximum_staff, Initial_staff"
      " + STEP(Staff_added, Week_staff_added)"
      " + Responsive_hiring * MAX(0, Staff_needed - Initial_staff))",
      "people"),
    V("Time to open posts", "2", "week"),
    V("Time to fill posts", "6", "week"),
    V("Assimilation time", "16", "week"),
    V("Completion share", "0.99", "fraction"),
    V("Project active", "IF(Work_done < Completion_share * Project_scope,"
      " 1, 0)", "dimensionless"),
]
case = build(WORK, PRESSURE, STAFF)
print(f"{len(case.stocks)} stocks, {len(case.flows)} flows, "
      f"{len(case.vars)} variables, {len(case.params)} constants")
base = case.run()
fin = finish_week(base)
print(f"as managed: true 99% in week {fin:.1f} (plan {PLAN_WEEKS}); "
      f"effort {base['Effort spent'].iloc[-1]:,.0f} person-weeks "
      f"(plan {PLAN_EFFORT:,}); staff peak {base['Total staff'].max():.1f}")

The full model has 8 stocks, 8 flows and 50 variables, 26 of them constants we can change in `run()`. As managed, the project is delivered in week 70.8, nearly 21 weeks late, and uses 2,383 person-weeks, 2.4 times the plan, with the team at the maximum of 40. The model has not been fitted to the reports: chapter 10 shows how. But it should reproduce the shape of the reference mode, and the next figure compares them.

In [ ]:
#| fig-alt: "Two panels comparing the model with the synthetic reports over 76 weeks. Left, reported progress: the model's curve and the synthetic reports both rise steadily and then crawl; the model reaches 90% a little later. Right, staff: the model hires from about week 20 up to 40 people; the reports rise later and stop near 38."
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(weeks, rep, label="reports (synthetic)", color="#D55E00", ls="--")
a1.plot(base.index, base["Reported progress"], label="model: reported",
        color="#0072B2", ls="-")
a1.plot(base.index, base["True progress"], label="model: truly done",
        color="#009E73", ls=":")
a2.plot(weeks, staff, label="reports (synthetic)", color="#D55E00", ls="--")
a2.plot(base.index, base["Total staff"], label="model", color="#0072B2")
a1.set(title="Progress", xlabel="time (week)",
       ylabel="progress (% of scope)", xlim=(0, 80))
a2.set(title="Staff", xlabel="time (week)", ylabel="staff (people)",
       xlim=(0, 80), ylim=(0, 42))
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()
mr = base["Reported progress"]
print(f"model: reported 90% in week {mr.index[mr >= 90][0]:.1f}, true 99% "
      f"in week {fin:.1f}; reports: 90% in week {w90}, "
      f"delivered in week {w_end}")

Figure: The model as managed against the synthetic status reports: progress (left) and staff (right). The model reproduces the three parts of the reference mode (steady reported progress that then crawls, staff added late, and delivery around week 70), though it hires a little earlier and reports a little less progress in the middle of the project than the reports do.

The model's reported progress reaches 90% in week 51.8, against week 50 in the reports, and it delivers in week 70.8 against week 66. The gap between the model's reported and true progress is the hidden rework that the reports cannot see. A model that matches the shape and is wrong in the details is enough for the questions of this chapter, which are about why and what if; a forecast of the delivery date would need calibration.

**Try it 3.**

- *Check.* In the work-only model the team of 20 can do 19 tasks a week (5% of its time goes on QA). How long would 1,000 tasks take with perfect quality, and where do the other weeks of the work-only run go?
- *Apply.* In the full model, set `discovery_time_without_qa` to 6, 12 and 24 weeks. How do the finish week and the effort change?
- *Extend.* Our managers plan on the visible work to do only. Suppose they could see the hidden rework too: replace `Required rate` with `(Work_to_do + Undiscovered_rework) / Time_left` using `with_equation`. Does seeing the truth help?

<details><summary>Worked answer</summary>

*Check.* All 1,000 tasks would take 1,000 / 19 = 52.6 weeks, and our finish line of 99% (990 tasks) 52.1 weeks, which is what the run with quality 1 gives. The work-only run needs 65.2 weeks: about 13 weeks go on finding and redoing the flawed 15%. *Apply.* The project finishes in week 62.5 with a discovery time of 6 weeks, 70.8 with 12 and 90.5 with 24, and the effort rises from 2,098 to 2,383 and 3,106 person-weeks. Finding flaws late is expensive twice over: the project lasts longer and a large team waits for the last flaws. *Extend.* No: the project finishes in week 71.5 with 2,452 person-weeks, slightly worse. Seeing more work raises the pressure earlier, which brings overtime, haste and fatigue earlier, so more rework is made. Better information helps only if the response to it is better too.
</details>

In [ ]:
# Check: where does the time go in the work-only model?
print(f"work only: {finish_week(d1):.1f} weeks; 1000 tasks / 19 tasks a "
      f"week = {1000 / 19:.1f} weeks")
# Apply: a slower discovery of flaws in the full model
for t in (6, 12, 24):
    d = case.run(discovery_time_without_qa=t)
    print(f"discovery time without QA {t:>2} weeks: true 99% in week "
          f"{finish_week(d):.1f}, effort {d['Effort spent'].iloc[-1]:,.0f}")
# Extend: management that plans on all the work, found or not
seen = case.with_equation(
    "Required rate", "(Work_to_do + Undiscovered_rework) / Time_left")
d = seen.run()
print(f"plans on hidden rework too: true 99% in week {finish_week(d):.1f},"
      f" effort {d['Effort spent'].iloc[-1]:,.0f}, staff peak "
      f"{d['Total staff'].max():.1f}")

## 4. Tests

Before we explain the behaviour, we test the model, as in chapter 9. Four tests: units on every element; **conservation**, here of work (the three work stocks must always add up to the scope, because work is only ever moved between them) and of people (everyone on the project either started on it or came through an open post); **extreme conditions**, inputs set to values far outside the normal range; and halving `dt`.

In [ ]:
dt = case.dt
# 1. Units on every element
print("missing units:", [e["name"] for e in case.stocks + case.flows
                         + case.vars if not e.get("units")])
# 2. Conservation: the three work stocks always add up to the scope,
#    and every person hired came through an open post
work = base[["Work to do", "Undiscovered rework", "Work done"]].sum(axis=1)
people = base[["Posts open", "New staff", "Experienced staff"]].sum(axis=1)
opened = 20 + dt * base["Opening posts"].cumsum().shift(fill_value=0)
print(f"work: largest gap from 1000 tasks {(work - 1000).abs().max():.1e}; "
      f"people: largest gap {(people - opened).abs().max():.1e}; "
      f"clipped: {base.attrs['clipped']}")
# 3. Extreme conditions
nobody = case.run(initial_staff=0, responsive_hiring=0)
perfect = case.with_equation("Quality", "1").run()
relaxed = case.run(deadline=1000)
huge = case.run(project_scope=10000)
print(f"nobody, no hiring: work done {nobody['Work done'].max():.1f}, "
      f"effort {nobody['Effort spent'].iloc[-1]:.1f}")
print(f"perfect quality: undiscovered rework never above "
      f"{perfect['Undiscovered rework'].max():.1f} tasks")
print(f"no deadline pressure: overtime max {relaxed['Overtime'].max():.2f},"
      f" staff max {relaxed['Total staff'].max():.1f}")
print(f"ten times the scope: staff max {huge['Total staff'].max():.1f}, "
      f"overtime max {huge['Overtime'].max():.2f}, quality min "
      f"{huge['Quality'].min():.3f}, clipped: {huge.attrs['clipped']}")
try:
    case.run(project_scope=0)
except lb.ModelError as err:    # the package explains what went wrong
    print("empty project:", err)
# 4. Halve dt
for step in (0.25, 0.125, 0.0625):
    d = case.run(dt=step)
    print(f"dt {step:<6}: true 99% in week {finish_week(d):6.2f}, effort "
          f"{d['Effort spent'].iloc[-1]:8.1f} person-weeks")

Every element has units. Work is conserved to about $10^{-12}$ tasks and people to about $10^{-13}$, and no stock was clipped at zero, so the books balance with no help from the floor. The extreme conditions behave: with nobody on the project and no hiring nothing is done and no effort is spent; with perfect quality no rework ever appears; with a deadline far away there is no overtime and no hiring; and with ten times the scope the team stops at its maximum of 40, overtime at its limit of 1.3, and quality at 0.623, all without a clip. Halving the time step from 0.25 to 0.125 to 0.0625 week moves the finish from week 70.75 to 70.75 to 70.69 and the effort by about 2 person-weeks: well below anything we report.

One test fails. An empty project (a scope of 0) stops the run at time 0 with a division by zero in `Reported progress`. It does not matter for this project, but a model that will be reused, perhaps inside a portfolio of projects, should not break on an empty one. The repair is a guard: an empty project is 100% complete.

In [ ]:
# Repair: progress of an empty project is 100%, not 0 / 0
case = (case
        .with_equation("Reported progress", "IF(Project_scope > 0, 100 * "
                       "(Work_done + Undiscovered_rework) / Project_scope,"
                       " 100)")
        .with_equation("True progress", "IF(Project_scope > 0, 100 * "
                       "Work_done / Project_scope, 100)"))
empty = case.run(project_scope=0)
print(f"empty project: reported {empty['Reported progress'].iloc[-1]:.0f}%,"
      f" effort {empty['Effort spent'].iloc[-1]:.1f}, staff max "
      f"{empty['Total staff'].max():.1f}")
same = case.run()
print("base run unchanged:",
      bool((same["Work done"] - base["Work done"]).abs().max() == 0))

The repaired model runs the empty project (100% complete, no effort spent, no hiring) and gives exactly the same numbers as before for our project. As in chapter 5, only an extreme test found the fault.

**Try it 4.**

- *Check.* With perfect quality, how much rework is ever discovered?
- *Apply.* Set the deadline to week 1, a date already missed when the project starts. Does anything break? What do managers do?
- *Extend.* Run the model with `method="rk4"` (chapter 1). Does the finish week change?

<details><summary>Worked answer</summary>

*Check.* None: 0.0 tasks, as there is nothing to find. *Apply.* Nothing breaks and nothing is clipped. With the deadline already gone, the time left is held at its 2-week minimum, the staff needed are enormous, and managers hire to the maximum at once: 34.5 people are on the project by week 10. The project finishes in week 56.5, earlier than any policy in section 6, but uses 2,525 person-weeks. It is an early hint of the result of section 6: hiring helps when it happens early. *Extend.* The finish week is 70.75 with both methods, so Euler with `dt` = 0.125 week is accurate enough.
</details>

In [ ]:
# Check: no flaws are ever found if none are made
print("perfect quality, rework discovered in total:",
      round(perfect["Discovering rework"].sum() * dt, 3))
# Apply: a deadline in the past (week 1) must not break anything
d = case.run(deadline=1)
print(f"deadline in week 1: true 99% in week {finish_week(d):.1f}, "
      f"effort {d['Effort spent'].iloc[-1]:,.0f}, staff "
      f"{d.loc[10.0, 'Total staff']:.1f} by week 10, "
      f"clipped {d.attrs['clipped']}")
# Extend: RK4 against Euler at dt = 0.125
d = case.run(method="rk4")
print(f"rk4: true 99% in week {finish_week(d):.2f} (Euler {fin:.2f})")

## 5. Explaining the behaviour

A model that reproduces the reference mode is a hypothesis that has survived a first test. Now we use it to explain the behaviour: which loops and which delays make the project late. We first look at the timing of the main variables, then switch mechanisms off one at a time.

In [ ]:
#| fig-alt: "Two panels over 100 weeks of the model as managed. Left, reported progress, true progress and the plan: reported progress reaches 90% close to the deadline in week 50, while true progress lags it by up to about 7 points and reaches 99% only around week 71. Right, schedule pressure, overtime and quality: pressure climbs steeply towards the deadline and collapses soon after it, overtime sits at its limit of 1.3 from about week 21 until just after the deadline, and quality falls from 0.85 to about 0.63."
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
show = base.loc[:100]
a1.plot(show.index, np.minimum(100, 100 * show.index / PLAN_WEEKS),
        label="plan")
a1.plot(show.index, show["Reported progress"], label="reported")
a1.plot(show.index, show["True progress"], label="truly done")
a1.set(title="Reported and true progress", xlabel="time (week)",
       ylabel="progress (% of scope)")
a2.plot(show.index, show["Schedule pressure"].clip(upper=3),
        label="schedule pressure (cut at 3)", color="#CC79A7", ls="-.")
a2.plot(show.index, show["Overtime"], label="overtime",
        color="#E69F00", ls=(0, (5, 1)))
a2.plot(show.index, show["Quality"], label="quality",
        color="#56B4E9", ls=(0, (3, 1, 1, 1)))
a2.set(title="Pressure, overtime and quality", xlabel="time (week)",
       ylabel="ratio (dimensionless)")
for a in (a1, a2):
    a.axvline(50, color="grey", lw=1, ls=":")
    a.legend(fontsize=8)
plt.tight_layout()
plt.show()
gap = base["Reported progress"] - base["True progress"]
print(f"reported minus true progress peaks at {gap.max():.1f} points in "
      f"week {gap.idxmax():.1f}")
for col in ("Overtime", "Haste", "New staff", "Undiscovered rework"):
    s = base[col]
    print(f"{col:<20} peaks at {s.max():6.2f} in week {s.idxmax():5.1f}")
print(f"quality is lowest, {base['Quality'].min():.3f}, in week "
      f"{base['Quality'].idxmin():.1f}")

Figure: The project as managed. Left, reported and true progress against the plan; right, schedule pressure (cut at 3 for display), overtime and quality, with the deadline marked at week 50. Notice that pressure, overtime and falling quality all come before the deadline, while the gap between reported and true progress is widest just after it.

The sequence is the dynamic hypothesis in action. The plan ignored the time that QA and coordination take, so the team starts behind and overtime rises from the first week, reaching its limit of 1.3 in week 21.5. Haste joins it and reaches its limit in week 39.6. Quality is lowest, 0.631, in week 48.4, just before the deadline, when the team is tired and rushing and the new hires (8.84 of them at the peak, in week 50.1) are still learning. The flaws made in those weeks surface after the deadline: `Undiscovered rework` peaks at 66.7 tasks in week 52.8, exactly when reported progress runs furthest ahead of true progress, by 6.7 points. That is the 90% syndrome explained: the work reported as done in the last weeks before the deadline was the worst work of the project.

To see which mechanisms matter most, we switch each off in turn. This is a **loop knockout** test (chapter 9): a constant or an equation is changed so that one link of a loop is cut. Setting `Time_to_fatigue` to a million weeks means fatigue never builds; replacing `Quality` with 1 removes every rework loop at once. `score` collects the measures a project board would ask for.

In [ ]:
def score(d):
    """Schedule and cost of one run, against the naive plan."""
    rp = d["Reported progress"]
    f = finish_week(d)
    return pd.Series({
        "finish (week)": f,
        "late (weeks)": f - PLAN_WEEKS,
        "effort (person-weeks)": d["Effort spent"].iloc[-1],
        "over budget (%)": 100 * (d["Effort spent"].iloc[-1]
                                  / PLAN_EFFORT - 1),
        "reported 90% (week)": rp.index[rp >= 90][0],
        "rework made (tasks)": d["Creating rework"].iloc[:-1].sum() * dt,
        "peak staff": d["Total staff"].max()})


# Switch off one mechanism at a time and see how much of the overrun goes
KNOCKOUTS = {
    "as managed": (case, {}),
    "no rework (quality 1)": (case.with_equation("Quality", "1"), {}),
    "no fatigue": (case, {"time_to_fatigue": 1e6}),
    "no overtime": (case, {"overtime_limit": 1}),
    "no haste": (case, {"haste_limit": 1}),
    "no hiring": (case, {"responsive_hiring": 0}),
    "rookies as good as veterans": (case, {"rookie_productivity": 1,
                                           "training_overhead": 0,
                                           "rookie_skill": 1}),
    "no communication overhead": (case, {
        "communication_overhead_per_person": 0}),
}
knock = pd.DataFrame({k: score(m.run(**p))
                      for k, (m, p) in KNOCKOUTS.items()}).T
print(knock.round(1).to_string())

Rework is the main cause of the overrun. Without it the project is 1.0 week late and 34.5% over budget, the cost of the overheads and the hiring the plan ignored; with it, 20.8 weeks late and 138% over. Fatigue comes next: without it the project finishes in week 60.5, halving the delay, because fatigue both slows the team and lowers quality. With no overtime at all the project finishes in week 65.5 and uses 1,707 person-weeks instead of 2,383: overtime is a fix that fails, as in chapter 7.

Two results go against intuition. Switching haste off makes the project *later* (week 72.5): in our model the extra speed of cutting corners outweighs its cost in quality, whose effect we set at a modest 0.3 per unit of haste. That is an assumption worth testing with data. And switching responsive hiring off delays delivery by only 1.6 weeks (72.4 against 70.8) while it saves 613 person-weeks: the late hires arrive when there is little left for them to do, and they bring overheads and errors with them. Removing the costs of a bigger team brings the finish forward: by 4.8 weeks if rookies are as good as veterans, by 3.6 weeks with no communication overhead. The mechanisms interact, so their effects do not add up: the knockouts are a ranking, not a budget.

**Try it 5.**

- *Check.* How many of the 20.8 weeks of delay remain when rework is knocked out?
- *Apply.* Knock out fatigue and haste together. Is the result the sum of the two knockouts?
- *Extend.* Switch off rework, fatigue, overtime, haste and hiring all at once. When does the project finish, and why is it still not week 50?

<details><summary>Worked answer</summary>

*Check.* 1.0 week: rework accounts for about 95% of the delay in this model. *Apply.* No. Fatigue alone brings the finish to week 60.5 and haste alone pushes it to 72.5, so adding the changes would give about 62.2; together they give 60.9 with 1,712 person-weeks. Without fatigue, the extra quality lost to haste is smaller, so the knockouts interact. *Extend.* Week 57.6. The team's normal capacity is 17.20 tasks a week, not the 20 of the plan, because 5% of its time goes on QA and about 9.5% on coordinating 20 people. Even a perfect project misses a plan that leaves out the time people spend on anything but tasks.
</details>

In [ ]:
# Check: the share of the delay explained by rework alone
late = knock["late (weeks)"]
print(f"rework knocked out: {late['no rework (quality 1)']:.1f} of "
      f"{late['as managed']:.1f} weeks late remain")
# Apply: two mechanisms at once, fatigue and haste
d = case.run(time_to_fatigue=1e6, haste_limit=1)
print(f"no fatigue and no haste: finish {finish_week(d):.1f}, effort "
      f"{d['Effort spent'].iloc[-1]:,.0f}")
# Extend: everything off but the overheads of the plan itself
ideal = case.with_equation("Quality", "1").run(
    time_to_fatigue=1e6, overtime_limit=1, haste_limit=1,
    responsive_hiring=0)
print(f"no rework, pressure or hiring: finish {finish_week(ideal):.1f}, "
      f"capacity {ideal['Normal capacity'].iloc[0]:.2f} tasks/week")

## 6. Policies and robustness

The board asks what it should do differently next time, or what it can still do on this project. We test four single policies, each a parameter setting as in chapter 11, and a package of all four:

| Policy | What changes in the model | Setting |
|---|---|---|
| More QA | `QA share` | 15% of staff time instead of 5% from the start; flaws are found in 3 weeks instead of 6 |
| More staff, early | `Responsive hiring`, `Staff added` | 10 people requested in week 0, and no further hiring to the deadline |
| No overtime | `Overtime limit` | 1: normal hours only |
| A realistic deadline | `Deadline` | the week the project as managed would reach with no deadline pressure at all |
| Package | all four | the realistic deadline is worked out again for the package |

In [ ]:
# The deadline the team would meet with no pressure at all
realistic = float(np.ceil(finish_week(case.run(deadline=1000))))
POLICIES = {
    "As managed": {},
    "More QA (15%)": {"qa_share": 0.15},
    "10 more staff in week 0": {"responsive_hiring": 0, "staff_added": 10},
    "No overtime": {"overtime_limit": 1},
    f"Deadline week {realistic:.0f}": {"deadline": realistic},
}
pkg = {"qa_share": 0.15, "responsive_hiring": 0, "staff_added": 10,
       "overtime_limit": 1}
pkg_deadline = float(np.ceil(finish_week(case.run(deadline=1000, **pkg))))
POLICIES[f"Package, deadline week {pkg_deadline:.0f}"] = dict(
    pkg, deadline=pkg_deadline)
runs = {k: case.run(**p) for k, p in POLICIES.items()}
board = pd.DataFrame({k: score(d) for k, d in runs.items()}).T
print(f"realistic deadline: week {realistic:.0f}; for the package: "
      f"week {pkg_deadline:.0f}\n")
print(board.round(1).to_string())

The realistic deadline is week 71, the date the team of 20 would deliver if nobody pushed it; for the package it is week 62. Each single policy beats the project as managed on effort, and all but the realistic deadline beat it on time. Adding 10 people at the start finishes in week 63.6, more QA in week 65.2 and no overtime in week 65.5. The realistic deadline delivers in week 71.0, almost the same week as the project as managed (70.8), but with 1,420 person-weeks instead of 2,383, the least effort of any policy: without pressure there is no overtime, no haste, no late hiring and far less rework (177 tasks instead of 399). The package delivers in week 61.6 with 1,769 person-weeks, 9 weeks sooner and about 600 person-weeks cheaper than the project as managed.

In [ ]:
#| fig-alt: "Two panels over 100 weeks for six policies. Left, work truly done: the package and adding staff at the start finish earliest; the project as managed finishes last but one. Right, cumulative effort: the project as managed spends the most, ending near 2,400 person-weeks; the realistic deadline spends the least, near 1,400."
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
for k, d in runs.items():
    a1.plot(d.index, d["True progress"], label=k)
    a2.plot(d.index, d["Effort spent"], label=k)
a1.set(title="Work truly done", xlabel="time (week)",
       ylabel="progress (% of scope)", xlim=(0, 100))
a2.set(title="Effort spent", xlabel="time (week)",
       ylabel="effort (person-weeks)", xlim=(0, 100))
a2.axhline(PLAN_EFFORT, color="grey", lw=1, ls=":")
a1.legend(fontsize=7)
plt.tight_layout()
plt.show()

Figure: Work truly done (left) and cumulative effort (right) under six policies, with the planned budget of 1,000 person-weeks dotted. The policies differ little in progress until week 40 and a great deal in effort, because a late finish with a large team is what makes a project expensive.

**Adding staff earlier or later.** The "more staff" policy asks for people in week 0. What if the same 10 people are asked for later, as usually happens once the delay is visible? The cell below requests them in weeks 0, 5, ..., 60, with no other hiring, and compares the result with adding no one.

In [ ]:
#| fig-alt: "Two panels: the week in which 10 extra people are requested, from 0 to 60, against the finish week (left) and the effort (right), with no responsive hiring. Adding them in week 0 finishes earliest; the later they are requested, the later the finish, and from about week 40 the project finishes later than with no one added. Effort is above the no-addition line throughout."
when = np.arange(0, 65, 5)
rows = []
for w in when:
    d = case.run(responsive_hiring=0, staff_added=10, week_staff_added=w)
    rows.append({"week requested": w, "finish (week)": finish_week(d),
                 "effort (person-weeks)": d["Effort spent"].iloc[-1]})
timing = pd.DataFrame(rows).set_index("week requested")
alone = case.run(responsive_hiring=0)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
a1.plot(timing.index, timing["finish (week)"], marker="o",
        label="10 people added")
a1.axhline(finish_week(alone), color="grey", ls=":", label="no one added")
a2.plot(timing.index, timing["effort (person-weeks)"], marker="o",
        label="10 people added")
a2.axhline(alone["Effort spent"].iloc[-1], color="grey", ls=":",
           label="no one added")
a1.set(title="When the project finishes", xlabel="week staff requested",
       ylabel="finish (week)")
a2.set(title="What it costs", xlabel="week staff requested",
       ylabel="effort (person-weeks)")
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"no one added: finish {finish_week(alone):.1f}, effort "
      f"{alone['Effort spent'].iloc[-1]:,.0f}")
print(timing.round(1).T.to_string())

Figure: The week in which 10 extra people are requested, against the week the project finishes (left) and the effort it uses (right), with the result of adding no one dotted. Every week of hesitation costs roughly a quarter of a week of delivery, and from week 40 the extra people make the project later.

With no one added, the fixed team of 20 delivers in week 72.4 with 1,770 person-weeks. Ten people requested in week 0 bring delivery forward to week 63.6. Requested in week 20 they bring it to 69.4, and in week 40 they make it *later*, 72.6; requested in week 50, at the deadline, they deliver in week 73.5, 1.1 weeks after the team that was left alone, and cost 184 extra person-weeks. This is what software engineers call Brooks's law, that adding people to a late project makes it later [@brooks1975]; the model shows when it holds. New staff need weeks to arrive and months to become fully productive, and in the meantime they take time from the experienced staff and add errors that surface later still. Early, there is time for the investment to pay back; late, there is not. Effort is higher than with no one added whenever the people come, so even early hiring is a trade of money for time.

**Will the ranking survive what we do not know?** Several parameters are guesses: normal quality, how long flaws take to find, the strength of fatigue, how productive and how quickly assimilated new staff are, and the communication overhead. Following chapter 11, we draw 200 plausible worlds at random and run every policy in every world, 1,200 runs in one call to `run_batch`.

In [ ]:
# 200 plausible worlds: the parameters we are least sure of, drawn at random
n = 200
worlds = pd.DataFrame({
    "normal_quality": rng.uniform(0.75, 0.95, n),
    "discovery_time_without_qa": rng.uniform(8, 16, n),
    "fatigue_strength": rng.uniform(0.3, 0.9, n),
    "rookie_productivity": rng.uniform(0.3, 0.7, n),
    "assimilation_time": rng.uniform(8, 24, n),
    "communication_overhead_per_person": rng.uniform(0.0025, 0.0075, n)})
# every policy in every world: assign() adds the policy's settings as columns
settings = pd.concat([worlds.assign(**p) for p in POLICIES.values()],
                     ignore_index=True)
settings = settings.fillna(pd.Series(case.params))  # defaults elsewhere
policy = np.repeat(list(POLICIES), n)       # policy name of each row
world = np.tile(np.arange(n), len(POLICIES))
out = case.run_batch(settings, include=["Work done", "Effort spent"])
done = out[out.variable == "Work done"].pivot(index="time", columns="run",
                                              values="value")
reached = done >= 990
# idxmax on True/False columns gives the first time each run reaches 99%
finish = reached.idxmax().where(reached.any())
effort = (out[out.variable == "Effort spent"].groupby("run")["value"]
          .last())
res = pd.DataFrame({"policy": policy, "world": world,
                    "finish": finish.values, "effort": effort.values})
print(f"{len(settings)} runs; unfinished by week 150: "
      f"{res.finish.isna().sum()}")
wide_f = res.pivot(index="world", columns="policy", values="finish")
wide_e = res.pivot(index="world", columns="policy", values="effort")
both = ((wide_f.lt(wide_f["As managed"], axis=0))
        & (wide_e.lt(wide_e["As managed"], axis=0)))
robust = pd.DataFrame({
    "finish, median": wide_f.median(),
    "finish, 90th pct": wide_f.quantile(0.9),
    "effort, median": wide_e.median(),
    "effort, 90th pct": wide_e.quantile(0.9),
    "beats as managed on both (%)": 100 * both.mean()}).loc[list(POLICIES)]
print(robust.round(1).to_string())

In [ ]:
#| fig-alt: "Two panels across 200 plausible worlds for six policies, each policy shown as a median dot on a bar from the 10th to the 90th percentile. Left, finish week: the project as managed and the week-71 deadline have the latest medians, near 70 and 71; the package and adding staff at the start the earliest, near 62. Right, effort: the project as managed is highest, near 2,400 person-weeks; the realistic deadline lowest, near 1,400."
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
ypos = np.arange(len(POLICIES))
for ax, wide, plan_line in ((a1, wide_f, PLAN_WEEKS),
                            (a2, wide_e, PLAN_EFFORT)):
    lo, mid, hi = (wide.quantile(q)[list(POLICIES)] for q in (0.1, 0.5, 0.9))
    ax.hlines(ypos, lo, hi, color="#0072B2", lw=3, ls="-",
              label="10th to 90th percentile")
    ax.plot(mid, ypos, "o", color="#D55E00", ls="none", label="median")
    ax.axvline(plan_line, color="grey", ls=":", label="plan")
a1.set_yticks(ypos, list(POLICIES))
a1.invert_yaxis()
a1.set(title="Finish across 200 worlds", xlabel="finish (week)")
a2.set(title="Effort across 200 worlds", xlabel="effort (person-weeks)")
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: Finish week (left) and effort (right) of six policies across 200 plausible worlds, as the median and the range from the 10th to the 90th percentile; the plan is dotted. The package is both the earliest and among the cheapest, and even its slow cases finish sooner than the typical case of the project as managed.

All 1,200 runs deliver within the 150 weeks simulated. The package delivers sooner and with less effort than the project as managed in all 200 worlds; so do adding staff at the start and banning overtime. More QA beats it on both counts in 57.5% of the worlds, and the realistic deadline in 43.5%, because the week-71 deadline is too tight in the less favourable worlds (its 90th percentile finish is week 82.0), and in those the pressure loops come back. The realistic deadline remains the cheapest policy, with a median of 1,420 person-weeks against 2,382 for the project as managed. The package also has the earliest slow cases: its median finish is week 62.0 and its 90th percentile week 67.9, against 70.0 and 79.4 as managed. It is not only better on average but safer to promise, which is what a board that has to commit to a date needs most.

**Try it 6.**

- *Check.* Which single policy uses the least effort in the central case?
- *Apply.* Instead of banning overtime, cap it at 1.1 or 1.2. How much of the benefit of the ban do you keep?
- *Extend.* QA is often cut when the deadline nears. Run QA at 15% until week 30 and at 5% after it, with `with_equation("QA share", "0.15 - 0.10 * STEP(1, 30)")`. Then try deadlines of weeks 50, 55, 60, 65 and 70. Is a deadline "a little more realistic" better than the naive one?

<details><summary>Worked answer</summary>

*Check.* The realistic deadline of week 71, with 1,420 person-weeks. *Apply.* Very little. With a cap of 1.1 the project finishes in week 67.1 with 1,933 person-weeks, and with 1.2 in week 69.0 with 2,180, against 65.5 and 1,707 with no overtime and 70.8 and 2,383 with the cap of 1.3. Every step of overtime allowed makes the project later and dearer. *Extend.* Early QA alone is worse than none: week 71.1 and 2,502 person-weeks, against 65.2 and 2,332 with QA kept at 15% throughout. Cutting QA as the deadline nears lengthens the discovery time just when the team is making the most errors. A deadline a little more realistic is also worse: week 55 gives delivery in week 73.9 and week 60 in week 75.4, both later than the naive week 50 (70.8). Only from week 65 does the effort fall far (1,495 person-weeks). A deadline that is still too tight brings the pressure late, when hiring and haste do the most harm; a deadline that is honest removes it.
</details>

In [ ]:
# Check: which single policy spends least, in the central case?
print("least effort:", board["effort (person-weeks)"].idxmin())
# Apply: a cap on overtime rather than a ban
for limit in (1.0, 1.1, 1.2, 1.3):
    d = case.run(overtime_limit=limit)
    print(f"overtime limit {limit}: finish {finish_week(d):.1f}, effort "
          f"{d['Effort spent'].iloc[-1]:,.0f}")
# Extend: QA at 15% only until week 30, then back to 5%
early_qa = case.with_equation("QA share", "0.15 - 0.10 * STEP(1, 30)")
for label, d in (("QA 15% throughout", runs["More QA (15%)"]),
                 ("QA 15% until week 30", early_qa.run())):
    print(f"{label:<22}: finish {finish_week(d):.1f}, effort "
          f"{d['Effort spent'].iloc[-1]:,.0f}")
# Extend: deadlines between the naive and the realistic
for dl in (50, 55, 60, 65, 70):
    d = case.run(deadline=dl)
    print(f"deadline week {dl}: finish {finish_week(d):.1f}, effort "
          f"{d['Effort spent'].iloc[-1]:,.0f}, peak staff "
          f"{d['Total staff'].max():.1f}")

## Tell the decision maker

> **The project was planned to take 50 weeks and 1,000 person-weeks. As it is being run, it will be delivered in about week 71 and cost about 2,400 person-weeks, 2.4 times the budget.**
>
> - *"90% done" is not 90% done.* Flaws made under pressure stay hidden until testing finds them, so the status reports reach 90% around the deadline and the last of the work then takes about 19 more weeks. Ask for the number of defects found each week, not only the percentage complete.
> - *Stop the overtime.* Tired people make more errors. Normal hours alone would deliver about 5 weeks sooner and save about 680 person-weeks.
> - *Add people at the start or not at all.* Ten people at the start would deliver in week 64. The same people requested at the deadline would make the project about a week later than adding no one, and still add to the cost.
> - *Set a date the team can meet.* A deadline of week 71 delivers in the same week as now with 40% less effort. A deadline that is only a little more realistic (week 55 or 60) is worse than either.
> - *Recommended: the four together.* More testing from the start, ten extra people at the start, no overtime and a deadline of week 62: delivery in week 62 for about 1,770 person-weeks. In all 200 versions of the project we simulated with different assumptions, this was both sooner and cheaper than the present approach.

## Check it

Habits for any project model:

- **Count the work.** The work stocks must add up to the scope at every step (section 4); if scope can change, add the changes to the count.
- **Define "done" before you report it.** The finish week depends on where the line is drawn, so check that the conclusions do not.
- **Know which constraints bind.** A cap that never binds is not a policy (chapter 7); print the maximum of every limited variable.
- **Halve `dt`** whenever you add a faster process, such as a shorter minimum task time.

The cell below moves the finish line from 99% of the scope to 98% and 99.5%.

In [ ]:
# Does the ranking depend on where we draw the finish line?
for share in (0.98, 0.99, 0.995):
    f = {k: finish_week(d, share=share) for k, d in runs.items()}
    order = sorted(f, key=f.get)
    print(f"{share:.1%} done: " + "; ".join(f"{k} {f[k]:.1f}"
                                            for k in order))

At every finish line the project as managed and the week-71 deadline come last, so the advice not to run the project as now is safe. The order at the top changes: at 98% adding staff at the start leads (week 58.6), while at 99% and 99.5% the package leads (weeks 61.6 and 64.0). A brief should say "the package, or early staff if only one change is possible", rather than name one winner by a fraction of a week.

## Going further

- **Slack when ahead.** Our overtime never falls below 1, so a project with a generous deadline does not slow down. Real teams often do (work expands to fill the time available); an effect of schedule pressure below 1 on the work rate adds this, and changes how generous a deadline should be.
- **Phases and concurrency.** Real projects run in phases, with flaws in design discovered during construction or testing. The Doing Work Cascade molecule of chapter 9 is a starting point; rework then flows back across phases, and starting a phase before the previous one is stable becomes a policy to test.
- **Loop dominance.** The knockouts of section 5 are a simple form of loop dominance analysis; @ford1999 gives a behavioural method for finding which loop dominates at each moment, which would show when the project switches from being driven by the balancing loops to the reinforcing ones.
- **Calibration and the hidden stock.** The undiscovered rework is never observed, but defect logs (flaws found each week) are the outflow of that stock. Chapter 10's methods can estimate normal quality and the discovery time from such logs, which would replace two of the guesses in section 6 with estimates.

## Mini-project

Turn the case into a **construction project**: a building with 2,000 tasks in two phases, design (800 tasks) then construction (1,200 tasks), with construction starting when design is reported 80% done.

1. Build two rework cycles, one per phase. Flaws in design are discovered partly in design (discovery time 8 weeks) and partly only during construction, where they also create rework in the construction tasks already built on them.
2. Give construction its own workforce, with a longer hiring delay (12 weeks) and a shorter assimilation time (6 weeks) than software.
3. Reproduce a reference mode of your own invention, labelled as synthetic, with a design phase that "finishes" on time and a construction phase that overruns.
4. Test the four policies of section 6, plus one of your own (for example, starting construction only when design is reported 95% done), across 200 plausible worlds, and write a five-bullet brief for the client.

## Further reading

- @sterman2000 discusses project dynamics, the rework cycle and the use of project models in practice at greater length.
- Hines, *Molecules of Structure* [@hines_molecules], describes the rework, schedule pressure, overtime, fatigue, quality and workforce structures our model is built from.
- @cooper1980 introduced the rework cycle in a shipbuilding dispute; @lyneis2007 survey three decades of system dynamics for project management.
- @abdelhamid1991 built the classic model of software projects, and @ford1998 model product development as linked phases, the structure of the mini-project.
- @martinezmoyano2013 collects modelling practices, including the testing habits of section 4.

To explore the rework structure on its own, the LoopBricks app has an interactive [Rework Cycle](https://waelrash1.github.io/loopbricks/#/m/reworkCycle).